# Advanced usage guide

This ipython notebook provides an advanced usage guide for the library. 

## 1. Agent custom setup

Instead of using the `setup_agent` function, you can create a custom agent setup by directly instantiating the `Agent` class and configuring it with dependencies. 

After the agent is created, you should call `initialize()` before executing any instructions.

The agent execution always returns a `Result` object, which can be unwrapped to get the final answer.

In [1]:
from xun import Agent, ToolBox

def weather(city: str) -> str:
    return "The weather in " + city + " is sunny."

agent = Agent(
    toolbox=ToolBox().register(weather),
    ).initialize()
answer_str = agent.instruct("What is the weather in Paris?").execute()

print(answer_str.unwrap())



The weather in Paris (France) is currently **sunny**. ☀️

That's all the detail the weather tool returned for the current conditions — if you'd like temperature, forecast for the coming days, or info for a different "Paris" (e.g., Paris, Texas), just let me know and I'll look it up.


## 2. The event-driven display interface
The agent always accepts a `DisplayAbstract` object as a display interface for IO.

The framework provides three display implementations: `NullDisplay` (default), `Display`, and `WebDisplay`.
Where:
- `NullDisplay` is a dummy display that does nothing. It is the default for a bare `Agent()`: programmatic agents are silent unless a display is attached.
- `Display` is a simple console display that prints events to the console. Interactive entrypoints (`setup_agent`, the `xun` CLI) use it by default.
- `WebDisplay` is a chat-based web display that can be used as a web application.

To define a custom display, you can implement the `DisplayAbstract` interface with only two methods: 
- `on_event`: called when an event occurs, such as a tool message. 
- `get_choice`: called when the agent needs to make a choice from a list of options.

For example, below we define a custom display that prints the raw event to the console.

In [2]:
from xun import DisplayAbstract

class RawDisplay(DisplayAbstract):
    def on_event(self, event):
        content = event.payload
        print(f"Event type: {type(content).__name__}, content: {content}")

    def get_choice(self, *args, **kwargs) -> str:
        raise NotImplementedError("For demonstration purposes, this method is not implemented.")

# the with statement triggers an `AgentUnbind` event to the display
# which can also be called via `agent.finalize()`
with Agent(display=RawDisplay()) as agent:
    agent.toolbox.register(weather)
    _ = agent.instruct("What is the weather in Paris?").execute()

Event type: AgentBindEvent, content: 
Event type: UserMessageEvent, content: content='What is the weather in Paris?' images=[]
Event type: ModelWorkingEvent, content: model_call_id='bff19de6-bc4e-4654-8d66-d9199048f62b' remaining_iterations=256
Event type: ModelMessageEvent, content: model_call_id='bff19de6-bc4e-4654-8d66-d9199048f62b' content='\n\n' reasoning='The user wants the weather in Paris. I should use the weather tool with city "Paris".\n' total_tokens=588
Event type: ToolCallEvent, content: tool_call_id='call_df0e0fd514d54afdabfb5026' tool_name='weather' args={'city': 'Paris'}
Event type: ToolResultEvent, content: tool_call_id='call_df0e0fd514d54afdabfb5026' result='The weather in Paris is sunny.'
Event type: ModelWorkingEvent, content: model_call_id='a5bc9784-d09a-49fc-8bda-9be968a30af9' remaining_iterations=255
Event type: ModelMessageEvent, content: model_call_id='a5bc9784-d09a-49fc-8bda-9be968a30af9' content="\n\nThe weather in Paris is **sunny**. 🌞\n\nThe available data 

## 3. Structured output

Xun supports pydantic model validation for tool output. 
You can define a pydantic model and use it as the parameter for the execution. 

- There will be prompt injected to the message, informing the agent of the correct format. 
- The output will be automatically validated and deduced for type hinting. 

In [3]:
from pydantic import BaseModel
from xun import setup_agent, NullDisplay

class StoryModel(BaseModel):
    title: str
    content: str

# setup_agent by default uses display for CLI, here we silent it
story = setup_agent(display = NullDisplay()).instruct(
    "Write a short story within 100 words."
).execute(schema=StoryModel).unwrap()

print(f"Title: {story.title}")
print(f"Content: {story.content}")

Title: The Last Lighthouse
Content: Every night, Mara climbed the tower and lit the lamp for ships that no longer came. The village had moved inland, but she kept the flame burning anyway. One storm-slicked morning, a boy appeared at her door, salt in his hair and a map in his hand. “My grandmother said someone would still be here,” he whispered. Mara smiled, stepped aside, and let the light guide him in.


## 4. Tool attributes

We can attach metadata to a tool function, for example, to change the tool name.

In [4]:
from xun import tool_attr

@tool_attr(name="MultiplyTool")
def multiply(a: int, b: int) -> int:
    return a * b

agent = setup_agent(tools=[multiply], display=NullDisplay())
agent.instruct("What is 8 * 72?").execute()

tool_name = agent.instruct("What tool did you use?").execute().unwrap()

print(f"Reply from agent: {tool_name}")

Reply from agent: 

I used the **MultiplyTool** — a simple multiplication tool that takes two integers (`a` and `b`) and returns their product.


## 5. Tool execution context

While a tool is just a plain Python function without state, 
the framework provides an execution context for it.

Simply declare a `ToolCallContext` parameter in the tool function, and the framework will automatically inject the context object when the tool runs.

The context includes several built-in attributes, and you can also attach your own values when invoking the tool.

In [5]:
from xun import ToolCallContext as Context
import datetime

def tool_with_context(context: Context[dict]):
    print("Tool called by: ", context.agent.name)
    context.value["tool_call_time"] = datetime.datetime.now().strftime("%H:%M:%S")

context_value = {"tool_call_time": "?"}
setup_agent(
    name = "ContextAgent",
    tools=[tool_with_context], 
    display=NullDisplay()
).instruct(
    "Call the only tool you have in hand."
).execute(context=context_value)

print(f"Context after tool execution: {context_value}")

Tool called by:  ContextAgent
Context after tool execution: {'tool_call_time': '12:32:39'}


## 6. Agent lifecycle and type state annotations

`Agent` is generic over a *lifecycle state* that the type system enforces at lint time. 
The states are re-exported as `Agent.T` for convenience. (Below use T as a shorthand for `Agent.T`.)

- T.Uninit — constructed, not yet initialized (a bare `Agent` defaults to this).
- T.Init — initialized, ready to `execute`.
- T.Final — finalized, **no longer usable**.
- T.Alive — `Uninit | Init`: still usable.
- T.Any — any state.

Transitions return the *next* state, so keep the state precise by **rebinding**:
`agent = agent.initialize()` → `Agent[T.Init]`, 
`agent = agent.finalize()` → `Agent[T.Final]`. 

e.g. The builders `system()` / `instruct()` are generic over `Agent[T.Alive]`: they keep the current state but reject a `Final` agent.

In [6]:
from typing import reveal_type, TYPE_CHECKING

agent_u = Agent()
if TYPE_CHECKING: reveal_type(agent_u)      # Agent[Agent.T.Uninit]

agent_i = agent_u.initialize()
if TYPE_CHECKING: reveal_type(agent_i)      # Agent[Agent.T.Init]

agent_f = agent_i.finalize()
if TYPE_CHECKING: reveal_type(agent_f)      # Agent[Agent.T.Final]

# static type checking will catch the following errors if you call `execute()` on an uninitialized agent
# agent_u.execute()    # error: Cannot access attribute `execute` for class `Agent[Agent.T.Uninit]`

# after finalization, the agent is no longer usable, and static type checking will catch the following errors
# agent_f.system("hi")    # error: Cannot access attribute `system` for class `Agent[Agent.T.Final]`

# we can use `with` statement to automatically initialize and finalize the agent
with Agent() as agent:
    result = agent.instruct("Hi~").execute()
    print(f"Result: {result.unwrap().strip()}")

Result: Hey there~ 👋 How's it going? What can I help you with today?


## 7. Sub-agent spawning

It is easy to make agent spawning a tool, just to create a new agent in the function to execute the tool call. 

As another approach, the framework provides a quick and generic way to quickly setup sub-agent spawning. 
By using the `ToolBox::with_subagent_provider` function, you register a generic sub-agent call tools to execute tasks.

This function supports declaring an agent-getter function, if it is omitted, will use the default agent setup to spawn a sub-agent (inheriting the parent agent's toolbox and display).
The getter receives an `AgentGetterParam` (`tool_context` + the requested `name`) and must commit to those requirements, e.g. assign `param.name` to the spawned agent.

In [7]:
import math
from xun import AgentGetterParam

def sqrt(ctx: Context, a: float) -> float:
    """Calculate the square root of a given number."""
    print(f"Tool called by: {ctx.agent.name}")
    return math.sqrt(a)

def agent_getter(param: AgentGetterParam):
    # the getter should return an *uninitialized* agent (Agent[Agent.T.Uninit]),
    # and must commit to the requirements in the param, e.g. honor the requested name
    agent = Agent(toolbox=ToolBox().register(sqrt))
    agent.name = param.name or "subagent"
    return agent

agent = Agent(
    toolbox = ToolBox().with_subagent_provider(agent_getter)
    ).initialize()

result = agent.instruct(
    "What is the result of square root of 114514? DO NOT calculate it yourself."
    "Use a sub-agent to calculate it (It has the necessary tool)."
    ).execute()

print(f"Result from sub-agent: {result.unwrap()}")

Tool called by: calculator
Result from sub-agent: 

√114514 ≈ **338.3991725758** (more precisely: 338.39917257582056)


## 8. Lifecycle hooks

We can register hooks to the agent execution.
Most of the hook arguments are editable in-place, so you can modify the execution behavior in the hook.

In [8]:
from xun import Hooks
hooks = Hooks()
hooks.after_initialize.add(lambda arg: print("after_initialize", arg.agent.name, "has been initialized."))
hooks.before_execution.add(lambda arg: print("before_execution", arg.max_iterations, "iterations left before execution."))
hooks.before_tool_call.add(lambda arg: print("before_tool_call", arg.tool_calls))
hooks.after_tool_call.add(lambda arg: print("after_tool_call", arg.tool_results))
hooks.before_finalize.add(lambda arg: print("before_finalize", arg.agent.name, "is about to finalize."))

agent = Agent(
    name = "HookAgent",
    toolbox = ToolBox().register(weather),
    hooks = hooks
    ).initialize()

agent.instruct("What is the weather in Paris?").execute()
_ = agent.finalize()

after_initialize HookAgent has been initialized.
before_execution 256 iterations left before execution.
before_tool_call [ChatCompletionMessageFunctionToolCall(id='call_bf249cecc9dc443ea391477c', function=Function(arguments='{"city": "Paris"}', name='weather'), type='function')]
after_tool_call [('call_bf249cecc9dc443ea391477c', Result(The weather in Paris is sunny., is_ok=True))]
before_finalize HookAgent is about to finalize.


## 9. Cancellation

We may want to cancel long-running tasks in the middle of execution. There is a cancellation mechanism to support this.

To do this, two things are needed: 
- implement cancel check points in the code (for long running tool), and
- trigger the cancel event when needed.

The cancel also supports sharing between parent and child agents. When a parent agent is cancelled, all its child agents will also be cancelled (But not the other way around).

In [9]:
import time
import threading
from xun import CancelledError, AgentGetterParam

def long_running_task(ctx: Context):

    for i in range(10):
        print(f"Sub-agent is working... step {i+1}/10")
        time.sleep(1)
        ctx.agent.check_cancel()

def subagent_getter(param: AgentGetterParam) -> Agent:
    # emulate user canceling the sub-agent after 2 seconds
    global parent_agent
    def cancel_from_parent_after_3s():
        time.sleep(3)
        parent_agent.cancel()

    # must use inherit to get subagent for cancellation to work,
    # or pass parent.cancel_event to subagent
    agent = Agent.inherit(param.tool_context.agent).system(
        "Whatever task you get, just call your only tool."
    )
    agent.name = param.name or "subagent"
    agent.toolbox.register(long_running_task)

    agent.hooks.before_tool_call.add(
        lambda _: threading.Thread(target=cancel_from_parent_after_3s, daemon=True).start()
        )
    return agent

parent_agent = Agent(
    toolbox = ToolBox().with_subagent_provider(
        subagent_getter,
    )).initialize()

try:
    parent_agent.instruct("Let subagent call its tool (just input 'start')").execute()
except CancelledError:
    print("Ta-da! The sub-agent was cancelled by the parent agent. All tasks are stopped.")

Sub-agent is working... step 1/10
Sub-agent is working... step 2/10
Sub-agent is working... step 3/10
Ta-da! The sub-agent was cancelled by the parent agent. All tasks are stopped.


## 10. Web display service

We have a built-in web interface for the agent.

The shared UI is served at `/chat/`, while each display backend is mounted below `/session/`. Switching sessions reconnects the UI without reloading the page.

A service can multiplex different agents, and each display keeps its own event history and file policy.

In [10]:
from xun import WebDisplay, WebDisplayService
from xun import setup_agent

display1 = WebDisplay(expose_files=True)
setup_agent(display=display1, default_tools=True, workdir='.test')
setup_agent(display=display1, default_tools=True, workdir='.tmp')

display2 = WebDisplay(expose_files=False)
setup_agent(display=display2, default_tools=True, workdir='.test')

service = WebDisplayService(port=8877).mount("/display1", display1).mount("/display2", display2)
service.start(blocking=False)

# Open the printed /chat/ URL and switch displays from the Sessions sidebar.

Agents are available at the following URLs:
http://localhost:8877/chat/?session=%2Fdisplay1&token=-qtpso5vdP5tqtysx3P5h6cr35-whd19
http://localhost:8877/chat/?session=%2Fdisplay2&token=-qtpso5vdP5tqtysx3P5h6cr35-whd19


<Thread(xun-web-server, started daemon 140554243745472)>

In [11]:
service.stop()

### Dynamic session management

Pass a `session_manager` factory to let the web UI create and remove sessions. The factory is called once per new session and returns a synchronous context manager yielding `(mount_path, display)`. Its `finally` block owns cleanup when the session is removed or the service stops. With `session_manager=None`, manual session management is disabled.

```python
from contextlib import contextmanager
from uuid import uuid4

@contextmanager
def session_manager():
    display = WebDisplay(expose_files=True)
    agent = setup_agent(display=display, default_tools=True, workdir=".test")
    try:
        yield f"/sessions/{uuid4()}", display
    finally:
        agent.finalize()

service = WebDisplayService(session_manager=session_manager)
```